# Step 1: Data Collection & Dataset Creation

In this notebook, we will download historical prices of the following assets from Yahoo Finance (from 2010-01-01 to 2025-12-31):
- **NIFTY 50 (`^NSEI`)**
- **Gold (`GC=F`)**
- **Crude Oil (`CL=F`)**
- **USD/INR (`INR=X`)**
- **India VIX (`^INDIAVIX`)**

We will clean column names, merge them on dates, inspect the merged dataset, and save both raw files and the master processed dataset.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import os

In [2]:
START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

In [3]:
print("Downloading NIFTY 50...")
nifty = yf.download("^NSEI", start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"NIFTY 50 Shape: {nifty.shape}")
nifty.head()

[*********************100%***********************]  1 of 1 completed

NIFTY 50 Shape: (3928, 5)


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2010-01-04,5232.200195,5238.450195,5167.100098,5200.899902,0
2010-01-05,5277.899902,5288.350098,5242.399902,5277.149902,0
2010-01-06,5281.799805,5310.850098,5260.049805,5278.149902,0
2010-01-07,5263.100098,5302.549805,5244.750000,5281.799805,0
2010-01-08,5244.750000,5276.750000,5234.700195,5264.250000,0


In [4]:
print("Downloading Gold...")
gold = yf.download("GC=F", start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"Gold Shape: {gold.shape}")
gold.head()

[*********************100%***********************]  1 of 1 completed

Gold Shape: (4022, 5)


Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2010-01-04,1117.699951,1122.300049,1097.099976,1117.699951,184
2010-01-05,1118.099976,1126.500000,1115.000000,1118.099976,53
2010-01-06,1135.900024,1139.199951,1120.699951,1135.900024,363
2010-01-07,1133.099976,1133.099976,1129.199951,1133.099976,56
2010-01-08,1138.199951,1138.199951,1122.699951,1138.199951,54


In [5]:
print("Downloading Crude Oil...")
oil = yf.download("CL=F", start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"Crude Oil Shape: {oil.shape}")
oil.head()

[*********************100%***********************]  1 of 1 completed

Crude Oil Shape: (4023, 5)


Price,Close,High,Low,Open,Volume
Ticker,CL=F,CL=F,CL=F,CL=F,CL=F
Date,,,,,
2010-01-04,81.510002,81.680000,79.629997,79.629997,263542
2010-01-05,81.769997,82.000000,80.949997,81.629997,258887
2010-01-06,83.180000,83.519997,80.849998,81.430000,370059
2010-01-07,82.660004,83.360001,82.260002,83.199997,246632
2010-01-08,82.750000,83.470001,81.800003,82.650002,310377


In [6]:
print("Downloading USD/INR...")
usd = yf.download("INR=X", start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"USD/INR Shape: {usd.shape}")
usd.head()

[*********************100%***********************]  1 of 1 completed

USD/INR Shape: (4164, 5)


Price,Close,High,Low,Open,Volume
Ticker,INR=X,INR=X,INR=X,INR=X,INR=X
Date,,,,,
2010-01-01,46.610001,46.645000,46.400002,46.400002,0
2010-01-04,46.287998,46.610001,46.223000,46.610001,0
2010-01-05,46.119999,46.287998,46.088001,46.287998,0
2010-01-06,45.720001,46.180000,45.700001,46.147999,0
2010-01-07,45.688000,45.877998,45.557999,45.737999,0


In [7]:
print("Downloading India VIX...")
vix = yf.download("^INDIAVIX", start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"India VIX Shape: {vix.shape}")
vix.head()

[*********************100%***********************]  1 of 1 completed

India VIX Shape: (3928, 5)


Price,Close,High,Low,Open,Volume
Ticker,^INDIAVIX,^INDIAVIX,^INDIAVIX,^INDIAVIX,^INDIAVIX
Date,,,,,
2010-01-04,23.639999,24.969999,23.600000,23.790001,0
2010-01-05,22.270000,23.219999,22.120001,22.980000,0
2010-01-06,22.120001,22.549999,21.770000,21.770000,0
2010-01-07,22.500000,22.930000,22.219999,22.219999,0
2010-01-08,22.570000,22.709999,22.190001,22.370001,0


In [8]:
# Keep only Close price for auxiliary datasets and rename
if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.droplevel(1)
if isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.droplevel(1)
if isinstance(oil.columns, pd.MultiIndex):
    oil.columns = oil.columns.droplevel(1)
if isinstance(usd.columns, pd.MultiIndex):
    usd.columns = usd.columns.droplevel(1)
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.droplevel(1)

gold_close = gold[['Close']].copy()
gold_close.columns = ['Gold']

oil_close = oil[['Close']].copy()
oil_close.columns = ['Oil']

usd_close = usd[['Close']].copy()
usd_close.columns = ['USD_INR']

vix_close = vix[['Close']].copy()
vix_close.columns = ['India_VIX']

# Rename NIFTY columns
nifty_renamed = nifty.rename(columns={
    'Open': 'NIFTY_Open',
    'High': 'NIFTY_High',
    'Low': 'NIFTY_Low',
    'Close': 'NIFTY_Close',
    'Volume': 'Volume'
})

print("Data cleaning and renaming completed.")

Data cleaning and renaming completed.


In [9]:
# Merge datasets
market_data = nifty_renamed.join(gold_close)
market_data = market_data.join(oil_close)
market_data = market_data.join(usd_close)
market_data = market_data.join(vix_close)

print(f"Merged Dataset Shape: {market_data.shape}")
market_data.head()

Merged Dataset Shape: (3928, 9)


,NIFTY_Close,NIFTY_High,NIFTY_Low,NIFTY_Open,Volume,Gold,Oil,USD_INR,India_VIX
Date,,,,,,,,,
2010-01-04,5232.200195,5238.450195,5167.100098,5200.899902,0,1117.699951,81.510002,46.287998,23.639999
2010-01-05,5277.899902,5288.350098,5242.399902,5277.149902,0,1118.099976,81.769997,46.119999,22.270000
2010-01-06,5281.799805,5310.850098,5260.049805,5278.149902,0,1135.900024,83.180000,45.720001,22.120001
2010-01-07,5263.100098,5302.549805,5244.750000,5281.799805,0,1133.099976,82.660004,45.688000,22.500000
2010-01-08,5244.750000,5276.750000,5234.700195,5264.250000,0,1138.199951,82.750000,45.518002,22.570000


In [10]:
# Basic verification
print("=== SHAPE ===")
print(market_data.shape)
print("\n=== NULL VALUE COUNT ===")
print(market_data.isnull().sum())
print("\n=== INFO ===")
market_data.info()
print("\n=== DESCRIBE ===")
print(market_data.describe())

=== SHAPE ===
(3928, 9)

=== NULL VALUE COUNT ===
NIFTY_Close      0
NIFTY_High       0
NIFTY_Low        0
NIFTY_Open       0
Volume           0
Gold           110
Oil            109
USD_INR          8
India_VIX       16
dtype: int64

=== INFO ===
<class 'pandas.DataFrame'>
DatetimeIndex: 3928 entries, 2010-01-04 to 2025-12-30
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   NIFTY_Close  3928 non-null   float64
 1   NIFTY_High   3928 non-null   float64
 2   NIFTY_Low    3928 non-null   float64
 3   NIFTY_Open   3928 non-null   float64
 4   Volume       3928 non-null   int64  
 5   Gold         3818 non-null   float64
 6   Oil          3819 non-null   float64
 7   USD_INR      3920 non-null   float64
 8   India_VIX    3912 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 306.9 KB

=== DESCRIBE ===
        NIFTY_Close    NIFTY_High     NIFTY_Low    NIFTY_Open        Volume  \
count   3928.000000   3928.0

In [11]:
# Save datasets
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

nifty.to_csv("../data/raw/nifty.csv")
gold.to_csv("../data/raw/gold.csv")
oil.to_csv("../data/raw/oil.csv")
usd.to_csv("../data/raw/usd.csv")
vix.to_csv("../data/raw/vix.csv")

market_data.to_csv("../data/processed/market_data.csv")
print("Saved all CSV files successfully!")

Saved all CSV files successfully!
